
# QoE analysis: QoE-constrained SciPy vs. no-QoE SciPy

This version uses `optimal_solutions_scipy.zip` as the baseline, so all 50 QoE scenarios—including the four \(F_{KL}=0.4\) cases—have exact solver-matched counterparts.

Definitions:

\[
h_{ij} = \mathbf 1\{j\in C,\ u_i^{(N)}-u_{ij}>\tau\},
\qquad \tau=0.2
\]

\[
v_i = \frac{1}{N}\sum_j h_{ij}r_{ij},
\qquad
V_{\mathrm{QoE}}=\sum_i p_i^{NF}v_i.
\]

The local QoE budget is \(\epsilon=0.1\). Network gain is

\[
G = CHR_{NF}-CHR_{BS}.
\]


In [6]:

from pathlib import Path
import json, zipfile, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QOE_ZIP = Path("qoe_optimal_solutions_E0.1_T0.2.zip")
BASELINE_ZIP = Path("optimal_solutions_scipy.zip")
LASTFM_U = Path("lastfm_U_matrix.csv")
MOVIELENS_U = Path("movielens1k_U_matrix.csv")

QOE_TOL = 0.2
QOE_EPS = 0.1


In [7]:

def load_u_csv(path):
    arr = pd.read_csv(path, header=None).to_numpy(float)
    if arr.shape[0] == arr.shape[1] + 1 and np.allclose(arr[0], np.arange(arr.shape[1])):
        arr = arr[1:]
    np.fill_diagonal(arr, 0.0)
    return arr

U = {
    "lastfm": load_u_csv(LASTFM_U),
    "movielens1k": load_u_csv(MOVIELENS_U),
}

pattern = re.compile(
    r"sim_results_U(?P<dataset>lastfm|movielens1k)"
    r"_pop(?P<pop>\d+)_a(?P<alpha>\d+(?:\.\d+)?)_N(?P<N>\d+)"
    r"_C(?P<C>\d+)_CP(?P<cp>[A-Za-z]+)_Q(?P<Q>\d+(?:\.\d+)?)"
    r"_L(?P<L>\d+)_F(?P<fmetric>max|TV|KL)(?P<fbound>\d+(?:\.\d+)?)"
)

def parse_name(name):
    m = pattern.search(Path(name).name)
    d = m.groupdict()
    for k in ("pop", "N", "C", "L"):
        d[k] = int(d[k])
    for k in ("alpha", "Q", "fbound"):
        d[k] = float(d[k])
    return d


In [8]:

def compute_metrics(result, params, tau=QOE_TOL, eps=QOE_EPS):
    u = U[params["dataset"]]
    N, C = params["N"], params["C"]

    R = np.asarray(result["R_NA"], float)
    p = np.asarray(result["pi_final_NA"], float)
    p_bs = np.asarray(result["pi_bs"], float)

    cache = np.argsort(p_bs)[-C:]
    chr_bs = float(p_bs[cache].sum())
    chr_nf = float(p[cache].sum())
    gain = chr_nf - chr_bs

    diff = p - p_bs
    fmax = float(np.max(np.abs(diff)))
    ftv = float(0.5 * np.sum(np.abs(diff)))
    mask = p_bs > 0
    fkl = (
        np.inf if np.any(mask & (p <= 0))
        else float(np.sum(p_bs[mask] * np.log(p_bs[mask] / p[mask])))
    )

    cutoff = np.sort(u, axis=1)[:, -N]
    unsafe = (cutoff[:, None] - u[:, cache]) > tau
    unsafe_frac = np.sum(R[:, cache] * unsafe, axis=1) / N
    unsafe_exposure = float(np.dot(p, unsafe_frac))

    qmax = np.sort(u, axis=1)[:, -N:].sum(axis=1)
    qor = np.sum(R * u, axis=1)
    qratio = np.divide(qor, qmax, out=np.full_like(qor, np.nan), where=qmax > 0)

    return {
        "chr_bs": chr_bs,
        "chr_nf": chr_nf,
        "gain": gain,
        "fmax_actual": fmax,
        "ftv_actual": ftv,
        "fkl_actual": fkl,
        "unsafe_exposure": unsafe_exposure,
        "unsafe_row_mean": float(np.mean(unsafe_frac)),
        "unsafe_row_median": float(np.median(unsafe_frac)),
        "unsafe_row_max": float(np.max(unsafe_frac)),
        "unsafe_rows_positive": int(np.sum(unsafe_frac > 1e-10)),
        "unsafe_rows_over_eps": int(np.sum(unsafe_frac > eps + 1e-8)),
        "qor_ratio_mean": float(np.nanmean(qratio)),
        "qor_ratio_min": float(np.nanmin(qratio)),
    }


In [9]:

records = []

with zipfile.ZipFile(QOE_ZIP) as zq, zipfile.ZipFile(BASELINE_ZIP) as zb:
    qnames = [n for n in zq.namelist() if n.endswith(".json")]
    bnames = {Path(n).name: n for n in zb.namelist() if n.endswith(".json")}

    for qname in qnames:
        baseline_name = Path(qname).name.replace("_qoeE0.1_qoeT0.2", "")
        if baseline_name not in bnames:
            continue

        params = parse_name(baseline_name)

        with zq.open(qname) as f:
            qres = json.load(f)
        with zb.open(bnames[baseline_name]) as f:
            bres = json.load(f)

        qm = compute_metrics(qres, params)
        bm = compute_metrics(bres, params)

        rec = {**params, "qoe_file": Path(qname).name, "baseline_file": baseline_name}
        rec.update({f"qoe_{k}": v for k, v in qm.items()})
        rec.update({f"base_{k}": v for k, v in bm.items()})

        rec["gain_retained_pct"] = (
            100 * qm["gain"] / bm["gain"] if abs(bm["gain"]) > 1e-12 else np.nan
        )
        rec["gain_loss_pct"] = (
            100 * (bm["gain"] - qm["gain"]) / bm["gain"]
            if abs(bm["gain"]) > 1e-12 else np.nan
        )
        rec["unsafe_reduction_pct"] = (
            100 * (bm["unsafe_exposure"] - qm["unsafe_exposure"])
            / bm["unsafe_exposure"]
            if bm["unsafe_exposure"] > 1e-12 else np.nan
        )
        rec["unsafe_abs_reduction"] = bm["unsafe_exposure"] - qm["unsafe_exposure"]
        records.append(rec)

results = pd.DataFrame(records).sort_values(
    ["dataset", "C", "fmetric", "fbound"]
).reset_index(drop=True)

print("Exact matched pairs:", len(results))


Exact matched pairs: 50


## Headline statistics

In [10]:

pd.Series({
    "median_gain_retained_pct": results["gain_retained_pct"].median(),
    "mean_gain_retained_pct": results["gain_retained_pct"].mean(),
    "minimum_gain_retained_pct": results["gain_retained_pct"].min(),
    "cases_retaining_at_least_95pct_gain": int((results["gain_retained_pct"] >= 95).sum()),
    "cases_with_no_measurable_gain_loss": int((results["gain_loss_pct"].abs() <= 1e-6).sum()),
    "median_unsafe_reduction_pct": results["unsafe_reduction_pct"].median(),
    "cases_with_lower_unsafe_exposure": int((results["unsafe_abs_reduction"] > 1e-9).sum()),
    "maximum_qoe_unsafe_exposure": results["qoe_unsafe_exposure"].max(),
})


median_gain_retained_pct               98.019654
mean_gain_retained_pct                 97.015440
minimum_gain_retained_pct              91.887123
cases_retaining_at_least_95pct_gain    36.000000
cases_with_no_measurable_gain_loss     17.000000
median_unsafe_reduction_pct            43.902104
cases_with_lower_unsafe_exposure       45.000000
maximum_qoe_unsafe_exposure             0.089169
dtype: float64

## Cases where the no-QoE solution exceeds the chosen 10% QoE budget

In [ ]:

high = results[results["base_unsafe_exposure"] > QOE_EPS]

pd.Series({
    "cases": len(high),
    "median_baseline_unsafe_exposure": high["base_unsafe_exposure"].median(),
    "median_qoe_unsafe_exposure": high["qoe_unsafe_exposure"].median(),
    "median_unsafe_reduction_pct": high["unsafe_reduction_pct"].median(),
    "minimum_unsafe_reduction_pct": high["unsafe_reduction_pct"].min(),
    "median_gain_retained_pct": high["gain_retained_pct"].median(),
    "minimum_gain_retained_pct": high["gain_retained_pct"].min(),
})


## \(F_{KL}=0.4\) scenarios

In [ ]:

results.loc[
    results["fmetric"] == "KL",
    [
        "dataset", "C", "fbound",
        "base_gain", "qoe_gain", "gain_retained_pct",
        "base_unsafe_exposure", "qoe_unsafe_exposure", "unsafe_reduction_pct",
    ],
].sort_values(["dataset", "C"])


## Summary by fairness metric

In [ ]:

results.groupby("fmetric").agg(
    cases=("gain_retained_pct", "size"),
    gain_retained_median=("gain_retained_pct", "median"),
    gain_retained_min=("gain_retained_pct", "min"),
    unsafe_reduction_median=("unsafe_reduction_pct", "median"),
    baseline_unsafe_median=("base_unsafe_exposure", "median"),
    qoe_unsafe_median=("qoe_unsafe_exposure", "median"),
)


## Summary by cache size

In [ ]:

results.groupby("C").agg(
    cases=("gain_retained_pct", "size"),
    gain_retained_median=("gain_retained_pct", "median"),
    gain_retained_min=("gain_retained_pct", "min"),
    unsafe_reduction_median=("unsafe_reduction_pct", "median"),
    baseline_unsafe_median=("base_unsafe_exposure", "median"),
    qoe_unsafe_median=("qoe_unsafe_exposure", "median"),
)



## Cases where the weighted unsafe-exposure metric increases

The constraint bounds each state locally; it does not minimize the global weighted unsafe-exposure metric. Therefore, if the no-QoE solution is already safely below the 10% budget, a different equally optimal LP vertex may have slightly larger global weighted exposure while still satisfying the QoE constraint.


In [ ]:

results.loc[
    results["unsafe_abs_reduction"] < -1e-9,
    [
        "dataset", "C", "fmetric", "fbound",
        "base_unsafe_exposure", "qoe_unsafe_exposure",
        "gain_retained_pct",
    ],
]


In [ ]:

results.to_csv("qoe_paired_metrics_vs_scipy_E0.1_T0.2.csv", index=False)
print("Saved qoe_paired_metrics_vs_scipy_E0.1_T0.2.csv")


## Publication plots

These plots use all exact SciPy-matched scenarios, including the four
$F_{KL}=0.4$ cases. Marker shapes distinguish fairness metrics:
circle = $F_{\max}$, square = $F_{TV}$, triangle = $F_{KL}$.
Both plotting cells save PDF and PNG files for LaTeX.


In [ ]:
print("Total exact matched pairs:", len(results))
print(results["fmetric"].value_counts().sort_index())

display(
    results.loc[
        results["fmetric"] == "KL",
        ["dataset", "C", "fbound",
         "base_unsafe_exposure", "qoe_unsafe_exposure",
         "unsafe_reduction_pct", "gain_retained_pct"],
    ].sort_values(["dataset", "C"])
)


### Figure 1 — Unsafe exposure before vs. after QoE

Each point is one matched scenario. The diagonal means no change. Points below
the diagonal have lower stationary-demand-weighted unsafe exposure after adding
the QoE safeguard. The dotted lines mark $\epsilon=0.1$.


In [ ]:
marker_map = {"max": "o", "TV": "s", "KL": "^"}
label_map = {"max": r"$F_{\max}$", "TV": r"$F_{TV}$", "KL": r"$F_{KL}$"}

fig, ax = plt.subplots(figsize=(6.4, 5.2))

for metric in ["max", "TV", "KL"]:
    group = results[results["fmetric"] == metric]
    if group.empty:
        continue
    ax.scatter(
        100 * group["base_unsafe_exposure"],
        100 * group["qoe_unsafe_exposure"],
        marker=marker_map[metric],
        label=label_map[metric],
        alpha=0.85,
    )

limit = 1.05 * max(
    100 * results["base_unsafe_exposure"].max(),
    100 * results["qoe_unsafe_exposure"].max(),
    10.0,
)

ax.plot([0, limit], [0, limit], linestyle="--", linewidth=1, label="No change")
ax.axvline(100 * QOE_EPS, linestyle=":", linewidth=1)
ax.axhline(100 * QOE_EPS, linestyle=":", linewidth=1)
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel(r"Unsafe exposure without QoE constraint, $V_{\mathrm{QoE}}$ (%)")
ax.set_ylabel(r"Unsafe exposure with QoE constraint, $V_{\mathrm{QoE}}$ (%)")
ax.set_title("Effect of the QoE-aware safeguard")
ax.legend()
fig.tight_layout()

fig.savefig("qoe_exposure_before_after.pdf", bbox_inches="tight")
fig.savefig("qoe_exposure_before_after.png", dpi=300, bbox_inches="tight")
plt.show()


### Figure 2 — QoE protection versus retained network gain

This plot keeps only the cases where the no-QoE solution has
$V_{\mathrm{QoE}}>\epsilon$. The x-axis is the percentage reduction in
unsafe exposure; the y-axis is the percentage of the original network gain
retained.


In [ ]:
active = results[results["base_unsafe_exposure"] > QOE_EPS].copy()

fig, ax = plt.subplots(figsize=(6.4, 5.2))

for metric in ["max", "TV", "KL"]:
    group = active[active["fmetric"] == metric]
    if group.empty:
        continue
    ax.scatter(
        group["unsafe_reduction_pct"],
        group["gain_retained_pct"],
        marker=marker_map[metric],
        label=label_map[metric],
        alpha=0.85,
    )

ax.axvline(50, linestyle=":", linewidth=1)
ax.axhline(95, linestyle=":", linewidth=1)
ax.set_xlabel("Reduction in unsafe QoE exposure (%)")
ax.set_ylabel("Network gain retained (%)")
ax.set_title("QoE protection versus retained network gain")
ax.legend()
fig.tight_layout()

fig.savefig("qoe_active_tradeoff.pdf", bbox_inches="tight")
fig.savefig("qoe_active_tradeoff.png", dpi=300, bbox_inches="tight")
plt.show()


### Optional diagnostic — annotate only the KL points

This version is useful to verify where the four $F_{KL}$ points landed without
cluttering the publication figure.


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.2))

for metric in ["max", "TV", "KL"]:
    group = results[results["fmetric"] == metric]
    if group.empty:
        continue
    ax.scatter(
        100 * group["base_unsafe_exposure"],
        100 * group["qoe_unsafe_exposure"],
        marker=marker_map[metric],
        label=label_map[metric],
        alpha=0.85,
    )

for _, row in results[results["fmetric"] == "KL"].iterrows():
    ax.annotate(
        f'{row["dataset"]}, C={int(row["C"])}',
        (100 * row["base_unsafe_exposure"], 100 * row["qoe_unsafe_exposure"]),
        xytext=(4, 4), textcoords="offset points", fontsize=8
    )

limit = 1.05 * max(
    100 * results["base_unsafe_exposure"].max(),
    100 * results["qoe_unsafe_exposure"].max(),
    10.0,
)
ax.plot([0, limit], [0, limit], linestyle="--", linewidth=1)
ax.axvline(100 * QOE_EPS, linestyle=":", linewidth=1)
ax.axhline(100 * QOE_EPS, linestyle=":", linewidth=1)
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel(r"Unsafe exposure without QoE constraint, $V_{\mathrm{QoE}}$ (%)")
ax.set_ylabel(r"Unsafe exposure with QoE constraint, $V_{\mathrm{QoE}}$ (%)")
ax.set_title(r"QoE exposure with $F_{KL}$ points annotated")
ax.legend()
fig.tight_layout()
plt.show()
